# 07 — EDA curta do painel CAR × t0
Nº de imóveis, eventos por t0 e taxa-base no recorte vitrine. **Exploratório**: geometria vigente do CAR; ver `docs/dados/dicionario.md` e `docs/pesquisa/07-gate-do-y.md`.
Requer `python -m agrorisk.pipeline --etapa todas --recorte vitrine`.

In [1]:
import pandas as pd
from pathlib import Path
pd.set_option('display.width', 160)
p = pd.read_parquet(Path('..')/'data'/'processed'/'painel_car_t0.parquet')
print(p.shape, p.cod_imovel.nunique(), 'CARs distintos')

(182454, 28) 31952 CARs distintos


In [2]:
t = p.groupby('ano_prodes').agg(n_cars=('Y_total','size'), y_total=('Y_total','sum'), y_agro=('Y_agro','sum'), y_ilegal=('Y_ilegal','sum'))
t['taxa_base_pct'] = (100*t.y_total/t.n_cars).round(2)
t

,n_cars,y_total,y_agro,y_ilegal,taxa_base_pct
ano_prodes,,,,,
2018,13727,85,76,85,0.62
2019,16556,155,133,155,0.94
2020,19276,267,232,267,1.39
2021,21672,103,84,103,0.48
2022,23856,243,216,243,1.02
2023,26106,93,80,92,0.36
2024,29309,129,116,125,0.44
2025,31952,137,119,135,0.43


In [3]:
# eventos por porte (módulos fiscais) no período de teste 2021–2024
te = p[p.ano_prodes.between(2021, 2024)]
te.groupby('classe_porte').agg(n=('Y_total','size'), eventos=('Y_total','sum')).assign(taxa_pct=lambda d: (100*d.eventos/d.n).round(2))

,n,eventos,taxa_pct
classe_porte,,,
4-15MF,8438,172,2.04
<=4MF,90385,303,0.34
>15MF,2120,93,4.39


In [4]:
# sem evento prévio no imóvel (36 m): base do critério primário
sem = te[te.hist_imovel_n_anos_36m == 0]
print('CAR×t0 sem evento prévio:', len(sem), 'eventos:', int(sem.Y_total.sum()), 'taxa %:', round(100*sem.Y_total.mean(), 2))

CAR×t0 sem evento prévio: 95485 eventos: 391 taxa %: 0.41


In [5]:
# lacunas de preenchimento (NA) e fração com geometria atualizada depois de t0 (diagnóstico)
print(p.isna().mean().round(3)[lambda s: s > 0])
print('geom atualizada após t0 (%):', round(100*p.diag_geom_atualizada_pos_t0.mean(), 1))

embargo_data_min    1.0
dtype: float64
geom atualizada após t0 (%): 93.0
